In [8]:
%load_ext autoreload
%autoreload 2

import spglib
import numpy as np
import nglview as nv
from ase.io import write
from ase.build import make_supercell
from monty.serialization import loadfn
from crystring.representation import get_representation, get_crystal_structure
from crystring.deduplicate import Deduplicator
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.ase import AseAtomsAdaptor
from doped.utils.symmetry import get_wyckoff_dict_from_sgn 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
entries = loadfn("/home/qfu1/data/MP_data.json.gz")

In [49]:
id = "mp-19"
entry = entries[id]
MP_structure = AseAtomsAdaptor.get_atoms(entry["structure"])

# structure = SpacegroupAnalyzer(entry["structure"],symprec=1e-2)
# lattice = structure.get_conventional_standard_structure().lattice
# uc = {'a': lattice.a, 'b': lattice.b, 'c': lattice.c, 'alpha': lattice.alpha, 'beta': lattice.beta, 'gamma': lattice.gamma}
# uc

{'a': 4.601352515923067,
 'b': 4.601352515923067,
 'c': 5.900062000998235,
 'alpha': 90.0,
 'beta': 90.0,
 'gamma': 119.99999999999999}

In [45]:
representation, unit_cell = get_representation(MP_structure, mute = False)

152
0
3a
representation: P 31 2 1 Te 3a 0.257 0.000 0.000 


In [46]:
print(representation)
print(unit_cell)

P 31 2 1 Te 3a 0.257 0.000 0.000 
{'a': 4.601352515923067, 'b': 4.601352515923067, 'c': 5.900062000998235, 'alpha': 90.0, 'beta': 90.0, 'gamma': 120.00000000000001}


In [43]:
crystal_from_str, basis = get_crystal_structure(representation, unit_cell, return_basis = True)
matcher = StructureMatcher()
if not matcher.fit(AseAtomsAdaptor.get_structure(crystal_from_str), AseAtomsAdaptor.get_structure(MP_structure)):
    print("dif")
else:
    print("same")
    
dedup = Deduplicator(tol = 5e-1)
results = dedup.deduplicate([MP_structure, crystal_from_str])
if len(results) == 1:
    print("same")
else:
    print("dif")

same
same


In [46]:
structure = MP_structure
lattice = structure.get_cell()
scaled_positions = structure.get_scaled_positions()
numbers = structure.get_atomic_numbers()
symmetry_dataset = spglib.get_symmetry_dataset(cell=(spglib.find_primitive((lattice, scaled_positions, numbers), symprec=8e-2)), symprec=8e-2)
std_positions = symmetry_dataset['std_positions']
std_mapping_to_primitive = symmetry_dataset["std_mapping_to_primitive"]
equivalent_atoms = symmetry_dataset["equivalent_atoms"]
wyckoffs = symmetry_dataset["wyckoffs"]
# symmetry_dataset

In [47]:
wyckoff_dict = get_wyckoff_dict_from_sgn(symmetry_dataset["number"])
print(wyckoff_dict.keys())

dict_keys(['4a', '4b', '8c', '8d', '8e', '16f'])


In [48]:
def check_positions(list1, list2, tol = 0.01):
    true = 0 
    found_positions = []
    for pos1 in list1:
        for pos2 in list2:
            if all(abs(p1 - p2) <= tol for p1, p2 in zip(pos1, pos2)):
                found_positions.append(pos1)
                break
    if found_positions:
        print("Positions found in list 2 with tolerance", tol, ":")
        for pos in found_positions:
            true += 1
            print(pos)
    else:
        print("No positions found in list 2 with tolerance", tol)
    print(true)

In [49]:
check_positions(basis, std_positions)

Positions found in list 2 with tolerance 0.01 :
[0 0 1/2]
[0 1/2 3/4]
[1/2 1/2 0]
[1/2 0 1/4]
[0 0 0]
[0 1/2 1/4]
[1/2 1/2 1/2]
[1/2 0 3/4]
[0.645000000000000 0.242000000000000 0.422000000000000]
[0.855000000000000 0.258000000000000 0.922000000000000]
[0.758000000000000 0.145000000000000 0.672000000000000]
[0.742000000000000 0.355000000000000 0.172000000000000]
[0.355000000000000 0.258000000000000 0.828000000000000]
[0.145000000000000 0.242000000000000 0.328000000000000]
[0.242000000000000 0.355000000000000 0.578000000000000]
[0.258000000000000 0.145000000000000 0.0780000000000000]
[0.145000000000000 0.742000000000000 0.922000000000000]
[0.355000000000000 0.758000000000000 0.422000000000000]
[0.258000000000000 0.645000000000000 0.172000000000000]
[0.242000000000000 0.855000000000000 0.672000000000000]
[0.855000000000000 0.758000000000000 0.328000000000000]
[0.645000000000000 0.742000000000000 0.828000000000000]
[0.742000000000000 0.855000000000000 0.0780000000000000]
[0.758000000000000

In [50]:
std_positions

array([[0.5       , 0.5       , 0.        ],
       [0.5       , 0.        , 0.25      ],
       [0.        , 0.        , 0.        ],
       [0.        , 0.5       , 0.25      ],
       [0.64517134, 0.24174602, 0.422259  ],
       [0.25825398, 0.64517134, 0.172259  ],
       [0.74174602, 0.35482866, 0.172259  ],
       [0.35482866, 0.75825398, 0.422259  ],
       [0.14517134, 0.24174602, 0.327741  ],
       [0.25825398, 0.14517134, 0.077741  ],
       [0.74174602, 0.85482866, 0.077741  ],
       [0.85482866, 0.75825398, 0.327741  ],
       [0.        , 0.        , 0.5       ],
       [0.        , 0.5       , 0.75      ],
       [0.5       , 0.5       , 0.5       ],
       [0.5       , 0.        , 0.75      ],
       [0.14517134, 0.74174602, 0.922259  ],
       [0.75825398, 0.14517134, 0.672259  ],
       [0.24174602, 0.85482866, 0.672259  ],
       [0.85482866, 0.25825398, 0.922259  ],
       [0.64517134, 0.74174602, 0.827741  ],
       [0.75825398, 0.64517134, 0.577741  ],
       [0.

In [51]:
basis

array([[0, 0, 1/2],
       [0, 1/2, 3/4],
       [1/2, 1/2, 0],
       [1/2, 0, 1/4],
       [0, 0, 0],
       [0, 1/2, 1/4],
       [1/2, 1/2, 1/2],
       [1/2, 0, 3/4],
       [0.645000000000000, 0.242000000000000, 0.422000000000000],
       [0.855000000000000, 0.258000000000000, 0.922000000000000],
       [0.758000000000000, 0.145000000000000, 0.672000000000000],
       [0.742000000000000, 0.355000000000000, 0.172000000000000],
       [0.355000000000000, 0.258000000000000, 0.828000000000000],
       [0.145000000000000, 0.242000000000000, 0.328000000000000],
       [0.242000000000000, 0.355000000000000, 0.578000000000000],
       [0.258000000000000, 0.145000000000000, 0.0780000000000000],
       [0.145000000000000, 0.742000000000000, 0.922000000000000],
       [0.355000000000000, 0.758000000000000, 0.422000000000000],
       [0.258000000000000, 0.645000000000000, 0.172000000000000],
       [0.242000000000000, 0.855000000000000, 0.672000000000000],
       [0.855000000000000, 0.758000

In [52]:
write(f"{id}-pred.cif", crystal_from_str)
write(f"{id}-true.cif", MP_structure)

In [62]:
structure.wrap()
# structure = make_supercell(crystal_from_str,3*np.eye(3))
view = nv.show_ase(crystal_from_str)
view.camera = 'orthographic'
view.add_unitcell()
view

NGLWidget()

In [64]:
structure.wrap()
# structure = make_supercell(crystal_from_str,3*np.eye(3))
view = nv.show_ase(crystal_from_str)
view.camera = 'orthographic'
view.add_unitcell()
view

NGLWidget()